# Modeling

Comparing Logistic Regression, Random Forest, and XGBoost on the cleaned dataset.
Using stratified CV throughout — class imbalance (6.7% default) means a random split would be misleading.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

TARGET = 'SeriousDlqin2yrs'

# features to drop — decided in notebook 02 (near-zero correlation with target)
DROP_COLS = ['monthly_debt', 'income_per_dependent']

## Load processed data

In [2]:
df = pd.read_csv('../data/processed/loan_engineered.csv')
print('Shape:', df.shape)

X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]

print('Features:', X.columns.tolist())
print('\nClass distribution:')
print(y.value_counts(normalize=True).round(3))

Shape: (149999, 15)
Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'total_past_due', 'has_past_due']

Class distribution:
SeriousDlqin2yrs
0    0.933
1    0.067
Name: proportion, dtype: float64


## Train / test split

`stratify=y` preserves the 93/7 class ratio in both splits.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Default rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

Train: (119999, 12) | Test: (30000, 12)
Default rate — train: 0.067 | test: 0.067


## Handle class imbalance — SMOTE

Apply SMOTE only on training data — never on test. The test set should reflect real-world distribution.

In [4]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print('After SMOTE:')
print(pd.Series(y_train_res).value_counts())

After SMOTE:
SeriousDlqin2yrs
0    111978
1    111978
Name: count, dtype: int64


## Scale features

Required for Logistic Regression (sensitive to feature scale).
Tree-based models (RF, XGBoost) don't need it but it doesn't hurt.

In [5]:
scaler = StandardScaler()

# fit on resampled train only — no leakage
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

# save scaler — needed later for API
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler saved.')

Scaler saved.


## Baseline — cross-validation across all three models

Quick CV before full training — tells us which direction to invest time in.

In [6]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train_res, cv=skf, scoring='roc_auc')
    cv_results[name] = {'mean_auc': round(scores.mean(), 4), 'std': round(scores.std(), 4)}
    print(f'{name:25s}  AUC: {scores.mean():.4f} (+/- {scores.std():.4f})')

Logistic Regression        AUC: 0.8910 (+/- 0.0008)
Random Forest              AUC: 0.9882 (+/- 0.0002)
XGBoost                    AUC: 0.9824 (+/- 0.0003)


## Train all three models on full training set

In [7]:
trained = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_res)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    trained[name] = model
    print(f'{name:25s}  Test AUC: {auc:.4f}')

Logistic Regression        Test AUC: 0.8463
Random Forest              Test AUC: 0.8309
XGBoost                    Test AUC: 0.8365


## Hyperparameter tuning — XGBoost with Optuna

Tuning the best-performing model. Optuna uses Bayesian optimization — smarter than grid search.

In [8]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'eval_metric': 'logloss',
        'random_state': 42,
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train_scaled, y_train_res, cv=skf, scoring='roc_auc')
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest AUC: {study.best_value:.4f}')
print('Best params:', study.best_params)


Best AUC: 0.9851
Best params: {'n_estimators': 446, 'max_depth': 8, 'learning_rate': 0.2338312543394669, 'subsample': 0.9652544579144143, 'colsample_bytree': 0.9832539861004118, 'min_child_weight': 8}


## Train final XGBoost with best params

In [9]:
best_xgb = XGBClassifier(
    **study.best_params,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
best_xgb.fit(X_train_scaled, y_train_res)

final_auc = roc_auc_score(y_test, best_xgb.predict_proba(X_test_scaled)[:, 1])
print(f'Final XGBoost Test AUC: {final_auc:.4f}')

joblib.dump(best_xgb, '../models/xgboost_final.pkl')
print('Model saved → models/xgboost_final.pkl')

Final XGBoost Test AUC: 0.8133
Model saved → models/xgboost_final.pkl


## Model comparison

| Model | CV AUC | Test AUC |
|---|---|---|
| Logistic Regression | 0.8910 | **0.8463** |
| Random Forest | 0.9882 | 0.8309 |
| XGBoost (baseline) | 0.9824 | 0.8365 |
| XGBoost (tuned) | 0.9851 | 0.8133 |

**Why the CV vs test gap?**
- SMOTE was applied before CV, so synthetic samples leaked into validation folds
- CV folds saw balanced 50/50 data; test set has the real 93/7 distribution
- Tree models overfit the synthetic patterns — LR didn't
- Tuned XGBoost actually performed worst (0.8133) — Optuna was optimizing a leaky metric
- Fix: wrap SMOTE inside a Pipeline so it only applies to each fold's training portion

**Winner on held-out test data: Logistic Regression (0.8463)**  
Moving forward with LR for evaluation and SHAP.